# WFC — Wealth Forecast
Stock data gathering, visualization, price prediction, and news sentiment analysis.
Uses Yahoo Finance for data and three ML algorithms (Linear Regression, Gradient Boosting, Random Forest).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, json, datetime as dtt
from os.path import expanduser

%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)

sys.path.insert(0, os.path.dirname(os.path.abspath('wfc_doc.ipynb')))

from wfc_config import wfc_config
from wfc_ds import StockDataGather, StockDataRead, StockDataPlot, StockNow, wfc_dpredict
from wfc_news import wfc_feed

print('WFC imports OK')

## 1. Load Configuration

In [ ]:
cfg_path = expanduser('~/.wfc')
with open(cfg_path) as f:
    cfg = wfc_config(f.read())

print('Portfolios:')
cfg.show()
print(f'\nData location: {cfg.data_location()}')
print(f'Owned tickers: {[x["ticker"] for x in cfg.purchased]}')

## 2. Parameters — edit these before running

In [ ]:
TICKER       = 'AAPL'            # stock to analyse
START_DATE   = '2022-01-01'      # YYYY-MM-DD  (data gathering start)
END_DATE     = '2024-12-31'      # YYYY-MM-DD  (data gathering end)
PREDICT_DATE = '2025-03-31'      # YYYY-MM-DD  (future price target)
DATA_DIR     = cfg.data_location()

## 3. Gather Historical Data from Yahoo Finance

In [ ]:
sdg = StockDataGather(TICKER, START_DATE, END_DATE, DATA_DIR)
sdg.gather(force=False)   # set force=True to re-download
print(f'CSV: {sdg.csv_path()}')

## 4. Data Summary

In [ ]:
sdr = StockDataRead(TICKER, DATA_DIR)
sdr.summary(PREDICT_DATE)

## 5. Visualize — Closing Price & Volume

In [ ]:
sdp = StockDataPlot(sdr.csv_path())
fig = sdp.closing()
fig.show() if hasattr(fig, 'show') else None

In [ ]:
fig = sdp.volume()

In [ ]:
fig = sdp.daily()

## 6. Price Prediction — Three Algorithms

In [ ]:
import pandas as pd

current = StockNow(TICKER).get('Close')
print(f'{TICKER} current price: ${current:.2f}' if current else 'Current price unavailable')
print(f'Predicting for:    {PREDICT_DATE}\n')

rows = []
for algo in ['linear', 'gbr', 'rfr']:
    sdr.analyze(algo)
    price = sdr.predict(PREDICT_DATE)
    row = {'Algorithm': algo, 'Predicted ($)': round(price, 2),
           'R2': round(sdr.r2, 3), 'RMSE': round(sdr.rmse, 2), 'MAE': round(sdr.mae, 2)}
    if current:
        row['Change (%)'] = round((price - current) / current * 100, 1)
    rows.append(row)

df_results = pd.DataFrame(rows)
ensemble   = df_results['Predicted ($)'].mean()
display(df_results)
print(f'\nEnsemble prediction: ${ensemble:.2f}')
if current:
    print(f'Ensemble change:     {(ensemble - current)/current*100:+.1f}%')

## 7. News & Sentiment Analysis

In [ ]:
feed = wfc_feed(TICKER)
feed.get_news()
feed.show_summary(TICKER)

In [ ]:
# Browse all articles with sentiment scores
import pandas as pd

if feed.articles:
    df_news = pd.DataFrame(feed.articles)[['published', 'compound', 'title', 'publisher', 'url']]
    df_news.columns = ['Published', 'Compound', 'Title', 'Publisher', 'URL']
    df_news = df_news.sort_values('Compound', ascending=False)
    pd.set_option('display.max_colwidth', 80)
    display(df_news)

## 8. Portfolio Analysis — Owned Stocks

In [ ]:
from datetime import timedelta

HORIZON_DAYS = 14   # forecast horizon
future_str   = (dtt.date.today() + timedelta(days=HORIZON_DAYS)).strftime('%Y-%m-%d')

print(f"Portfolio overview — forecast horizon: {future_str}  ({HORIZON_DAYS} days)")
print('=' * 70)

for x in cfg.purchased:
    tkr = x['ticker']
    print(f'\n{tkr}')

    # Purchase summary
    total_shares, total_cost = 0, 0.0
    for purchase in x['ticker_purchase']:
        for date, details in purchase.items():
            n, p = details[0], details[1]
            total_shares += n
            total_cost   += n * p
            print(f'  {date}: {n} shares @ ${p:.2f}')
    print(f'  Total: {total_shares} shares, cost ${total_cost:,.0f}')

    # Current & predicted price
    current = StockNow(tkr).get('Close')
    if current:
        mkt_val = current * total_shares
        gain    = mkt_val - total_cost
        print(f'  Current: ${current:.2f}  mkt ${mkt_val:,.0f}  ({gain:+,.0f})')

    try:
        results  = wfc_dpredict(cfg, tkr, future_str).predict()
        if 'ensemble' in results:
            ens = results['ensemble']['price']
            chg = f"{(ens - current)/current*100:+.1f}%" if current else 'N/A'
            print(f'  Predicted ({future_str}): ${ens:.2f}  ({chg})')
    except Exception as e:
        print(f'  Prediction unavailable: {e}')

    # Sentiment
    try:
        f2 = wfc_feed(tkr)
        f2.get_news()
        if f2.articles:
            print(f'  News sentiment: {f2.overall_sentiment().upper()}  '
                  f'(+{f2.get_pos():.2f} / -{f2.get_neg():.2f} / ~{f2.get_neu():.2f})')
    except Exception as e:
        print(f'  News unavailable: {e}')